# 5.4 Eight people, and how they type

One claim, taken all the way through [05.1](05.1-relationships.ipynb)'s grid:

> **People are more identifiable by *how* they write than by *what* they write about.**

Mechanism: plausible, and it predicts something specific — habits should be stable while
subject matter drifts. Evidence: what this notebook builds. Replication: there is a four-year
gap in the corpus to check it on.

The eight busiest people on `#ubuntu-uk`. Action lines (`/me`) are dropped: they have a
different grammar and would be a free giveaway.

In [ ]:
import numpy as np
import pandas as pd
from goad_toolkit.datatransforms import (
    CountValues,
    Filter,
    Head,
    Pipeline,
    RegexFeature,
    TransformBase,
)
from goad_toolkit.visualizer import (
    Annotate,
    BarPlot,
    GroupedBarPlot,
    HeatmapPlot,
    HighlightCategory,
    PlotSettings,
    ScatterPlot,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from scripts.pipelines import build_irc_pipeline
from wa_analyzer.data import load_showcase

msgs = build_irc_pipeline().apply(load_showcase("ubuntu_irc"))
uk = (
    Pipeline()
    .add(Filter, expr="channel == '#ubuntu-uk' and not is_action")
    .add(RegexFeature, name="caps", column="message", pattern=r"^[A-Z]",
         feature="starts_upper", mode="has")
    .apply(msgs)
)
uk["length"] = uk.message.str.len()
uk["night"] = uk.hh.between(0, 5)

top8 = Pipeline().add(CountValues, column="author").add(Head, n=8).apply(uk)
people = uk[uk.author.isin(top8.author)].copy()

baseline = top8.n.iloc[0] / top8.n.sum()
print(top8.to_string(index=False))
print(f"\n{len(people):,} messages, {people.author.nunique()} authors")
print(f"guessing the most frequent author every time = {baseline:.1%}")

In [ ]:
people.head()

## 5.4.1 The classifier is an instrument, not the result

Fit a model that reads one message and names its author. **The accuracy is a gate, not a
finding** — it tells you whether there is anything to look at. What the model *learned* is the
finding, and that lives in the weights.

In [ ]:
train_msgs, test_msgs = train_test_split(
    people, test_size=0.25, random_state=42, stratify=people.author)

words = TfidfVectorizer(min_df=5, sublinear_tf=True)
one_liner = LogisticRegression(max_iter=1000, C=5)
one_liner.fit(words.fit_transform(train_msgs.message), train_msgs.author)

one_message_accuracy = one_liner.score(words.transform(test_msgs.message), test_msgs.author)
print(f"{len(words.vocabulary_):,} features, one message at a time")
print(f"accuracy {one_message_accuracy:.1%}  against a {baseline:.1%} baseline")

vocabulary = np.array(words.get_feature_names_out())
for i, author in enumerate(one_liner.classes_):
    strongest = vocabulary[np.argsort(one_liner.coef_[i])[::-1][:8]]
    print(f"{author:12s} {', '.join(strongest)}")

Twice the baseline is a gate comfortably passed and nowhere near a destination — half the
messages are still attributed to the wrong person. Read the rows, and they are not all the same
kind of thing:

- `ali1234` — `n900`, `qml`, `pidgin`, `mythtv`. **Subject matter.**
- `MartijnVdS` — `mungbean`, `dimpy`, `neuro`. **Other people's nicknames.** Not a writing
  habit at all; a social position.
- `zmoylan-pi` — `ireland`, `dublin`. **Location.**
- `daftykins` — `0o`, `xd`, `hrmm`. **Typing.**
- `popey` — `dont`, `didnt`, `thats`. **Typing** — specifically, missing apostrophes.

Only the last two are the claim. The others are a topic detector wearing a stylometry costume,
and a topic detector fails the moment somebody changes subject — which is exactly the situation
an authorship model exists for.

> **The verification step, and the habit worth stealing:** when a model works, read its
> weights and ask *what each one actually is*. A model can be right for a reason that does not
> generalise, and the accuracy will never tell you.

So stop using the model to find the fingerprint. Measure the habits directly, and let the model
come back later as a check.

## 5.4.2 Four habits, measured

Nothing here needs machine learning. Five `RegexFeature` steps and a `groupby` — literally the
same transform lesson 1 pointed at URLs, pointed at smileys and apostrophes instead. The one
non-regex habit is free: lesson 1's `mentions` step already extracted who a message addresses,
so "opens with someone's nick" is whether that extraction matched.

Start with smileys, where there are three ways to write the same thing.

In [ ]:
habits = (
    Pipeline()
    .add(RegexFeature, name="nosed", column="message",
         pattern=r"[:;=]-[)DPp(\]]", feature="nosed", mode="count")
    .add(RegexFeature, name="noseless", column="message",
         pattern=r"[:;=][)DPp(\]]|\bXD\b", feature="noseless", mode="count")
    .add(RegexFeature, name="smiley", column="message",
         pattern=r"[☺☻☹㋛]", feature="unicode_smiley", mode="count")
    .add(RegexFeature, name="apos_drop", column="message",
         pattern=r"(?i)\b(?:dont|doesnt|didnt|cant|wont|isnt|im|thats|its|ive|youre)\b",
         feature="apos_drop", mode="count")
    .add(RegexFeature, name="apos_keep", column="message",
         pattern=r"(?i)\b(?:don't|doesn't|didn't|can't|won't|isn't|i'm|that's|it's|i've|you're)\b",
         feature="apos_keep", mode="count")
    .apply(people)
)
habits["addresses"] = habits.addressed_to.notna()

per_author = habits.groupby("author")
smileys = per_author[["nosed", "noseless", "unicode_smiley"]].sum()
dialect = smileys.div(smileys.sum(axis=1), axis=0) * 100
dialect.columns = pd.Index(["nosed :-)", "noseless :)", "unicode ☺"])

dialects = PlotSettings(
    figsize=(11, 4),
    title="Three ways to write a smiley, and nobody mixes them",
    xlabel="",
    ylabel="share of that author's smileys (%)",
    xtick_rotation=30,
)
fig, ax = GroupedBarPlot(dialects).plot(
    data=dialect.reset_index().melt(id_vars="author", var_name="dialect", value_name="share"),
    x="author", y="share", hue="dialect",
    palette={"nosed :-)": "crimson", "noseless :)": "#bbbbbb", "unicode ☺": "steelblue"},
)
_ = ax.legend(title="", loc="upper left", bbox_to_anchor=(1.01, 1))

Three dialects, and **nobody mixes them**. `zmoylan-pi` and `diddledan` write the nose almost
every time, the other six essentially never — there is nobody in between. That is not what a
habit usually looks like. Most measurable differences between people are matters of degree;
this is a **discrete choice**, made once and held for five years, by people reading each
other's messages every day. `popey` is the third dialect on his own: ☺, a character the others
never type once.

The other habits — apostrophes, capitals, addressing someone, posting at night — go into one
fingerprint table, drawn as a heatmap so the eight rows compare at a glance.

In [ ]:
fingerprint = pd.DataFrame({
    "nosed :-)": dialect["nosed :-)"],
    "unicode ☺": dialect["unicode ☺"],
    "no apostrophe": per_author.apos_drop.sum()
    / (per_author.apos_drop.sum() + per_author.apos_keep.sum()) * 100,
    "starts capital": per_author.starts_upper.mean() * 100,
    "addresses": per_author.addresses.mean() * 100,
    "00-05h": per_author.night.mean() * 100,
})

marks = PlotSettings(figsize=(9, 4.5), title="Eight fingerprints (% of that author's messages)",  # ty: ignore[invalid-argument-type]
                     xlabel="", ylabel="")
fig, ax = HeatmapPlot(marks).plot(data=fingerprint, annot=True, fmt=".0f", cmap="rocket_r")

Every column has the same shape as the smileys: a couple of people at one extreme and
everybody else clustered at the other. `popey` and `foobarry` drop a third of their
apostrophes, everyone else almost none. `daftykins` does 15% of his talking between midnight
and 05:00; `foobarry` has posted in that window exactly zero times in five years. `MartijnVdS`
opens half his messages with someone's nick — the thing the classifier found, seen properly:
not that he *writes* differently, but that his role in the channel is answering people.

The channel average hides all of this. There is no such thing as a typical `#ubuntu-uk` author.

## 5.4.3 Why the classifier only reached 47%, when the habits look this decisive

Both of those are true at once, and the reason is the unit of analysis.

In [ ]:
has_smiley = habits[["nosed", "noseless", "unicode_smiley"]].sum(axis=1) > 0
has_contraction = habits[["apos_drop", "apos_keep"]].sum(axis=1) > 0
print(f"messages containing a smiley       {has_smiley.mean():.1%}")
print(f"messages containing a contraction  {has_contraction.mean():.1%}")
print(f"messages containing neither        {(~has_smiley & ~has_contraction).mean():.1%}")

**Three quarters of messages carry no fingerprint at all.** The habits are near-deterministic
*when they appear*, and most single lines are `ok`, `thanks`, `brb`. A model reading one line
usually has nothing to go on, so 47% is what a strong signal looks like when it is sparse.

The fix is to stop asking about one message. Pool fifty of them per person and ask again — and
since "what counts as one row" is a decision worth defending, it is a transform of its own,
defined here where it can be read. Then two models on the same blocks: one that knows every
word, one that knows ten habits and nothing else.

In [ ]:
class BlockMessages(TransformBase):
    """Pool each author's shuffled messages into fixed-size blocks: habit means, text joined."""

    def transform(self, data: pd.DataFrame, features: list[str], size: int = 50,
                  seed: int = 0) -> pd.DataFrame:
        shuffled = data.sample(frac=1, random_state=seed)
        shuffled["block"] = shuffled.groupby("author").cumcount() // size
        grouped = shuffled.groupby(["author", "block"])
        blocks = grouped[features].mean()
        blocks["text"] = grouped.message.apply(" ".join)
        return blocks[grouped.size() == size].reset_index()


FEATURES = ["nosed", "noseless", "unicode_smiley", "apos_drop", "apos_keep",
            "starts_upper", "addresses", "night", "length", "n_question"]

blocks = Pipeline().add(BlockMessages, features=FEATURES, size=50).apply(habits)
train, test = train_test_split(blocks, test_size=0.25, random_state=42, stratify=blocks.author)

vec = TfidfVectorizer(min_df=3, sublinear_tf=True)
vocabulary_model = LogisticRegression(max_iter=1000, C=5)
vocabulary_model.fit(vec.fit_transform(train.text), train.author)

scaler = StandardScaler().fit(train[FEATURES])
habit_model = LogisticRegression(max_iter=2000)
habit_model.fit(scaler.transform(train[FEATURES]), train.author)

scores = pd.DataFrame({
    "model": ["baseline", "1 message, words", "50 messages, words", "50 messages, habits"],
    "accuracy": [
        baseline,
        one_message_accuracy,
        vocabulary_model.score(vec.transform(test.text), test.author),
        habit_model.score(scaler.transform(test[FEATURES]), test.author),
    ],
})
gate = PlotSettings(
    figsize=(9, 4),
    title=f"{len(blocks):,} blocks of 50 messages: ten numbers do what {len(vec.vocabulary_):,} words do",
    xlabel="",
    ylabel="accuracy on held-out data",
    base_color="#cccccc",
    highlight_color="crimson",
)
bars = BarPlot(gate)
fig, ax = bars.plot(data=scores, x="model", y="accuracy", color=gate.base_color)
_ = bars.plot_on(HighlightCategory(gate), categories=["50 messages, habits"])
print(scores.round(3).to_string(index=False))

**Ten numbers do what seventeen thousand words do.** Both block models are at or near ceiling,
and the one that knows nothing about vocabulary — no topics, no nicknames, no place names — is
within a point of the one that knows everything. That is the claim, demonstrated: identity is
in the habits, not the subject matter. It took volume to see, which is the honest caveat that
goes in the same sentence.

And because there are ten features rather than seventeen thousand, the weights are readable:
each row is a person, in the model's own words.

In [ ]:
weights = pd.DataFrame(habit_model.coef_, index=habit_model.classes_, columns=pd.Index(FEATURES))
weights.round(1)

`MartijnVdS` — **addresses**, everything else near zero: the role, again. `daftykins` —
**noseless** and **night**: types `:)` and is awake at 3am. `foobarry` — **night**, strongly
negative: identified largely by *never* being there. `popey` — unicode, apostrophes dropped,
short. Compare that with the tf-idf model's rows in §5.4.1: same task, same data, and one of
them can be read aloud.

## 5.4.4 Leg three: does it hold when nobody is looking?

The corpus spans five years. Fit nothing — just measure the same habits in 2013–2014 and again
in 2016–2017, and see whether people are still themselves. Four habits, one panel each: an
author on the diagonal is the same person in both halves.

In [ ]:
early = habits[habits.date.dt.year <= 2014]
late = habits[habits.date.dt.year >= 2016]
print(f"early {len(early):,} messages (2013-2014), late {len(late):,} (2016-2017)")

checks = [("addresses by nick (%)", "addresses", "mean", 100),
          ("starts with a capital (%)", "starts_upper", "mean", 100),
          ("median message length", "length", "median", 1),
          ("asks a question (%)", "n_question", "mean", 100)]

pairs = {}
for label, column, how, scale in checks:
    pair = pd.concat([early.groupby("author")[column].agg(how) * scale,
                      late.groupby("author")[column].agg(how) * scale], axis=1).dropna()
    pair.columns = pd.Index(["2013-2014", "2016-2017"])
    pairs[label] = pair

stable = PlotSettings(
    figsize=(11, 9),
    title="The same eight people, four years apart",
    subplot_titles=[f"{label}\nspearman {pair['2013-2014'].corr(pair['2016-2017'], method='spearman'):.2f}"
                    for label, pair in pairs.items()],
    max_cols=2,
)
host = ScatterPlot(stable)
fig, axes = host.create_figure(n_plots=4)
for ax, (label, pair) in zip(axes, pairs.items()):
    host.plot_on_axes(ScatterPlot(stable), ax, data=pair, x="2013-2014", y="2016-2017",
                      color="steelblue", s=60)
    lo, hi = pair.min().min(), pair.max().max()
    ax.plot([lo, hi], [lo, hi], color="lightgrey", linestyle="--", linewidth=1)
    for author, row in pair.iterrows():
        host.plot_on_axes(Annotate(stable), ax, text=str(author), xy=(row["2013-2014"], row["2016-2017"]),
                          xytext=(row["2013-2014"], row["2016-2017"]), arrow=False, fontsize=8)

Three of the four panels put everyone near the diagonal — spearman 0.93 to 0.98: the ordering
of eight people on a habit is unchanged after a four-year gap, on messages nobody was thinking
about when the habit formed. Leg three, cleanly.

The fourth panel is the useful one. **Question rate is not a trait** — the points scatter off
the line, spearman 0.55. Whether you ask questions depends on whether you currently have a
problem, and that changes; it moves people around the ranking while the typing habits hold
them still. A finding that came with its own counter-example is more believable than one that
did not, because it shows the measurement could have said no.

## 5.4.5 And now the objection

Run it through the grid honestly and it lands top-left: mechanism, evidence, replication. But
there is a sentence you cannot write.

**There are eight people here.**

The evidence is 218,000 messages, and every test above is at the author level for exactly that
reason. But `n = 8`, and the claim — *people are identified by how they type* — is about
people in general. A spearman of 0.98 across eight points is eight points. This is the most
seductive finding in the course and therefore the best place to say it: **large data does not
fix a small n at the unit your claim is about.** Lesson 2's pseudoreplication, arriving at the
end of the notebook that spent its whole length being careful.

What it would take to fix: more authors. The habits are cheap to measure — every cell above runs
on anybody with thirty messages — so the honest next step is to recompute the stability
correlations across every author with enough history in both periods, and report *that*
number instead of this one.

> **What survives, stated the way it should be reported.** *"Among the eight most active
> `#ubuntu-uk` authors, hand-picked typing habits classify blocks of 50 messages at over 99%,
> matching a 17,000-feature bag of words, and the habit rankings are preserved across a
> four-year gap (spearman 0.93–0.98) while question rate is not (0.55). Whether this holds for
> less active authors is untested."*

Every clause in that sentence is doing work, and none of it is the word "significant".

## Your turn

Your chat has people in it. Pick two habits you can write as a regex — a smiley dialect, a
catchphrase, a language switch, an apostrophe — and measure them per author with the pipeline
above. Then the two questions this notebook asked: does a model that only knows those habits
tell people apart on blocks of messages, and does each person's habit hold in the first and
the second half of the chat? Report the n at the author level, whatever it is.